##### Getting data via API from OpenStreetMap (Geoapify), cleaning, preparing final dataset of point of attractions (poi) and metro stations. + clean csv file

In [2]:
import requests
import pandas as pd

In [3]:
# getting data from geoapify api (key is generated for free on their website)

api_key = "756a595056e14418a79776989273ece7"

url = "https://api.geoapify.com/v2/places"

params = {
    "categories": "tourism.attraction",
    # bounding box for Barcelona
    "filter": "rect:2.05,41.47,2.23,41.32",
    "limit": 500,
    "apiKey": api_key
}

response = requests.get(url, params=params)

response.status_code

200

In [4]:
data_attractions = response.json()

# checking the number of attractions retrieved
len(data_attractions["features"])

500

In [5]:
# retrieving the next 500 attractions (if any)

params = {
    "categories": "tourism.attraction",
    "filter": "rect:2.05,41.47,2.23,41.32",
    "limit": 500,
    "offset": 500,
    "apiKey": api_key
}

response_2 = requests.get(url, params=params)

data_attractions_2 = response_2.json()

len(data_attractions_2["features"])
# got 221 attractions, so total is 500 + 221 = 721

221

In [ ]:
# getting all attractions in one list

all_attractions = (
    data_attractions["features"] +
    data_attractions_2["features"]
)

len(all_attractions)

721

In [7]:
# checking the first attraction in the list 
all_attractions[0]

{'type': 'Feature',
 'properties': {'country': 'Spain',
  'country_code': 'es',
  'state': 'Catalonia',
  'county': 'Barcelonès',
  'province': 'Barcelona',
  'city': 'Santa Coloma de Gramenet',
  'postcode': '08923',
  'district': 'Districte V',
  'neighbourhood': 'Raval',
  'suburb': 'Sant Andreu',
  'quarter': 'el Bon Pastor',
  'street': 'Parc Fluvial del Besòs',
  'iso3166_2': 'ES-CT',
  'iso3166_2_sublevel': 'ES-B',
  'lon': 2.2069359,
  'lat': 41.4430032,
  'state_code': 'CT',
  'formatted': 'Parc Fluvial del Besòs, 08923 Santa Coloma de Gramenet, Spain',
  'address_line1': 'Parc Fluvial del Besòs',
  'address_line2': '08923 Santa Coloma de Gramenet, Spain',
  'categories': ['tourism',
   'tourism.attraction',
   'tourism.attraction.artwork',
   'tourism.attraction.artwork.mural'],
  'details': ['details.artwork'],
  'datasource': {'sourcename': 'openstreetmap',
   'attribution': '© OpenStreetMap contributors',
   'license': 'Open Database License',
   'url': 'https://www.openst

In [8]:
# creating a dataframe with the attractions data

rows = []

for item in all_attractions:
    properties = item["properties"]

    rows.append({
        "name": properties.get("name"),
        "latitude": properties.get("lat"),
        "longitude": properties.get("lon"),
        "category": "attraction"
    })

df_attractions = pd.DataFrame(rows)

df_attractions.head()

,name,latitude,longitude,category
0,NaN,41.443003,2.206936,attraction
1,NaN,41.449042,2.195413,attraction
2,Ⓐ,41.464660,2.203062,attraction
3,NaN,41.460651,2.177291,attraction
4,NaN,41.448733,2.186346,attraction


In [9]:
df_attractions.shape

(721, 4)

In [10]:
df_attractions.isna().sum()

name         372
latitude       0
longitude      0
category       0
dtype: int64

In [ ]:
# getting data for metro stations

params = {
    "categories": "public_transport.subway",
    "filter": "rect:2.05,41.47,2.23,41.32",
    "limit": 500,
    "apiKey": api_key
}

response_metro = requests.get(url, params=params)

response_metro.status_code

200

In [ ]:
# checking the number of metro stations retrieved

data_metro = response_metro.json()

len(data_metro["features"])

174

In [ ]:
# creating a dataframe with the metro stations data

rows = []

for item in data_metro["features"]:
    properties = item["properties"]

    rows.append({
        "name": properties.get("name"),
        "latitude": properties.get("lat"),
        "longitude": properties.get("lon"),
        "category": "metro"
    })

df_metro = pd.DataFrame(rows)

df_metro.head()

,name,latitude,longitude,category
0,Fondo,41.451673,2.218408,metro
1,Llefià,41.441303,2.217548,metro
2,Torras i Bages,41.443127,2.190714,metro
3,Roquetes,41.447355,2.175783,metro
4,Torre Baró | Vallbona,41.459218,2.179945,metro


In [14]:
df_metro.shape

(174, 4)

In [15]:
df_metro.isna().sum()

name         0
latitude     0
longitude    0
category     0
dtype: int64

In [ ]:
# combining the two dataframes (attractions + metro) in one dataframe

df_poi = pd.concat(
    [df_attractions, df_metro],
    ignore_index=True
)

In [17]:
df_poi.shape

(895, 4)

In [18]:
df_poi["category"].value_counts()

category
attraction    721
metro         174
Name: count, dtype: int64

In [19]:
df_poi.duplicated().sum()

np.int64(0)

In [ ]:
# final clean dataframe

df_poi.head(10)

,name,latitude,longitude,category
0,NaN,41.443003,2.206936,attraction
1,NaN,41.449042,2.195413,attraction
2,Ⓐ,41.464660,2.203062,attraction
3,NaN,41.460651,2.177291,attraction
4,NaN,41.448733,2.186346,attraction
5,NaN,41.448257,2.176741,attraction
6,Mirador del Portell de Valldaura,41.447761,2.136969,attraction
7,Besart. The River Museum,41.442287,2.207409,attraction
8,El mirador,41.378568,2.054944,attraction
9,NaN,41.378670,2.053876,attraction


In [21]:
# df_poi.to_csv("barcelona_poi.csv", index=False)